# セミナーII 第2回 演習2：GitHub のコードを Colab で動かす

研究では、論文の著者が GitHub に公開したコードを動かすところから始めることがよくあります。このノートブックは、その手順をひと通りなぞる「型」です。記入例として **Depth Anything V2**（1枚の写真から奥行きを推定するモデル）を動かします。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. まず記入例のまま、上のセルから順に実行します。
4. 次に、自分で選んだリポジトリで試します。見出しに **（書き換える）** と付いた節のセルを直し、また上から順に進めます。直す行には `# ← 書き換える` と書いてあります。

リポジトリの選び方、README の読み方、つまずいたときの対処は、[ガイド（guide.md）](https://github.com/r-terasawa457/seminar2-2026/blob/lesson02/lesson02/guide.md) にまとめています。各節の見出しの下に、時間の目安を書いています。

**中断して再開するとき**：しばらく操作しないと Colab の接続が切れ、取得したリポジトリ、インストールしたライブラリ、重みが消えます（コピーに残るのは、ノートブックの文章、コード、出力です）。1. から順に実行し直してください。直したコードを残す方法は 9. にあります。

## 0. 手順の地図

目安：約3分

| 手順 | やること | このノートブック |
| --- | --- | --- |
| 計算環境を確かめる | Python、PyTorch、GPU の版を見る | 1. |
| リポジトリを決める | URL と版（コミット）を決める | 2.（書き換える） |
| 取得して README を読む | clone して、README の要点を探す | 3. |
| 依存ライブラリを入れる | requirements.txt から、必要なものだけ入れる | 4.（書き換える） |
| 学習済みの重みを取得する | 決められた場所に置く | 5.（書き換える） |
| 公式の例を動かす | README のコマンドをそのまま実行する | 6.（書き換える） |
| 自分のデータで試す | 手元の画像を入れる | 7. |
| Python から呼んで応用する | 結果の数値を別の処理につなげる | 8. |
| コードを直して変更を残す | 直したところをパッチとして保存する | 9. |
| 記録する | 何をして、どこでつまずいたか | 10. |

### 記入例：Depth Anything V2

- **何ができるか**：写真1枚から、各画素がどれくらい近いか（相対的な奥行き）を推定します。
- **選んだ理由**：重みがゲートなし（申請不要）で取得でき、一番小さい Small（約2500万パラメータ）は無料の GPU で数秒で動きます。画像を読むエンコーダは ViT（DINOv2）で、講義の Transformer がそのまま使われています。
- **リポジトリ**：[DepthAnything/Depth-Anything-V2](https://github.com/DepthAnything/Depth-Anything-V2)　論文：L. Yang et al., "Depth Anything V2," NeurIPS 2024（[arXiv:2406.09414](https://arxiv.org/abs/2406.09414)）
- **ライセンス**：コードと Small の重みは Apache-2.0。Base、Large の重みは CC BY-NC 4.0（非商用に限る）です。

### README のコマンドを Colab で実行するには

README のコマンドは、端末（ターミナル）で打つ形で書かれています。Colab のセルでは、次のように書き換えます。

| README の書き方 | Colab のセル | 理由 |
| --- | --- | --- |
| `git clone ...`、`python run.py ...` | `!git clone ...`、`!python run.py ...` | `!` を付けると、端末のコマンドとして実行される |
| `cd Depth-Anything-V2` | `%cd Depth-Anything-V2` | `!cd` は、その1行の中でしか効かない |
| `pip install ...` | `%pip install ...` | ノートブックが使っている Python に確実に入る |

`!` の行では、`{REPO_DIR}` のように波かっこで囲むと、Python の変数の値が埋め込まれます。

## 1. 計算環境を確かめる

目安：約2分

README の「動作環境」（Requirements、Installation など）と比べるために、この環境の版を表示します。Colab の Python や PyTorch の版は、時期によって変わります。

In [ ]:
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
START_DIR = Path.cwd()  # 最初の作業フォルダ（Colab では /content）

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__, "/ CUDA", torch.version.cuda)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name, f"/ メモリ {props.total_memory / 2**30:.1f} GiB")
else:
    print("GPU    : なし（「ランタイム」→「ランタイムのタイプを変更」で GPU を選び、最初から実行し直す）")
print(f"ディスクの空き: {shutil.disk_usage(START_DIR).free / 2**30:.0f} GiB")

device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. リポジトリを決める（書き換える）

目安：約2分（リポジトリを選ぶ時間は除く）

`REPO_REF` には、動かす版（コミット）を書きます。GitHub のリポジトリは日々更新されるので、版を決めておくと、後で同じ結果を再現できます。コミットは、GitHub のリポジトリのページで、ファイル一覧の上にある時計のアイコン（Commits）から確かめられます。最初は `None`（最新）で試し、動いたら 3. で表示されるコミットを書いておくのがおすすめです。

In [ ]:
REPO_URL = "https://github.com/DepthAnything/Depth-Anything-V2.git"  # ← 書き換える
REPO_REF = "a561b849ebae10a6f5ef49e26c83cbbcd36c71bf"  # ← 書き換える（コミット。None なら最新）

REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
REPO_DIR = START_DIR / REPO_NAME
# 9. で直したコードを保存する場所（Colab では Google ドライブ）
SAVE_DIR = Path("/content/drive/MyDrive/seminar2_lesson02") if IN_COLAB else START_DIR / "saved"
PATCH_FILE = SAVE_DIR / f"{REPO_NAME}.patch"
print("取得先:", REPO_DIR)

## 3. 取得して README を読む

目安：約10分（うち計算 約1分）

リポジトリを取得（clone）し、その中に移動します。以降のセルは、リポジトリのフォルダの中で実行します（README のコマンドは、ふつうリポジトリの中で実行する前提で書かれています）。

In [ ]:
if not REPO_DIR.exists():
    !git clone {REPO_URL} "{REPO_DIR}"
if REPO_REF:
    !git -C "{REPO_DIR}" checkout -q {REPO_REF}
%cd {REPO_DIR}
!git log -1 --oneline
# リポジトリの中の .py を import できるようにする
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

RESTORE_PATCH = False  # True：9. で保存した変更を当て直す（再開したとき）
if RESTORE_PATCH:
    if IN_COLAB and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    !git apply "{PATCH_FILE}"
    !git status --short

取得したファイルは、左の「ファイル」（フォルダのアイコン）から開けます。README の本文は GitHub のページのほうが読みやすいので、ここでは見出しの一覧と、環境に関係するファイルを表示します。

In [ ]:
readme = next((p for p in sorted(Path(".").iterdir()) if p.name.lower().startswith("readme")), None)
if readme:
    print(f"--- {readme.name} の見出し ---")
    for line in readme.read_text(encoding="utf-8").splitlines():
        if line.startswith("#"):
            print(line)

print("\n--- ファイルとフォルダ ---")
print(sorted(p.name + ("/" if p.is_dir() else "") for p in Path(".").iterdir() if not p.name.startswith(".")))

# Python の版の条件と、依存ライブラリの一覧
for name in ["pyproject.toml", "setup.py", "requirements.txt", "environment.yml"]:
    path = Path(name)
    if path.exists():
        text = path.read_text(encoding="utf-8")
        found = re.findall(r".*(?:requires-python|python_requires|python=).*", text)
        print(f"\n--- {name} ---")
        print("\n".join(found) if found and name != "requirements.txt" else text[:800])

README では、次の5つを探します。記入例で見つかったものを右に書きます。

| 探すもの | よくある見出し | 記入例（Depth Anything V2） |
| --- | --- | --- |
| 動作環境 | Requirements、Installation | 版の指定なし。`requirements.txt` に torch、opencv-python、gradio など |
| インストール | Installation、Setup、Preparation | `pip install -r requirements.txt` |
| 学習済みの重み | Pre-trained Models、Checkpoints、Model Zoo | 表のリンクから取得し、`checkpoints` フォルダに置く |
| 使い方 | Usage、Inference、Demo、Quick Start | `python run.py --encoder ... --img-path ... --outdir ...` |
| ライセンス | License | コードと Small は Apache-2.0、Base 以上の重みは非商用 |

`run.py` の `--encoder` の初期値は `vitl`（Large、約1.3GB、非商用）です。README の例をそのまま写すと Large を使うことになるので、6. では `--encoder vits`（Small）を指定します。

## 4. 依存ライブラリを入れる（書き換える）

目安：約5分（うち計算 約2分）

`requirements.txt` をそのまま入れると、Colab に入っている PyTorch が別の版に入れ替わり、GPU で動かなくなることがあります。Colab にすでに入っているもの（torch、torchvision など）と、使わないもの（記入例では、Web デモ用の gradio）を外して入れます。

- インストールの最後に「セッションを再起動」の案内が出たら、「ランタイム」→「セッションを再起動」を選び、1. から実行し直します（入れたライブラリは残ります）。
- `setup.py` や `pyproject.toml` があり、README が `pip install -e .` を指示しているリポジトリでは、下のセルの最後の行の代わりに `%pip install -e .` を実行します。

In [ ]:
SKIP = {"torch", "torchvision", "gradio", "gradio_imageslider"}  # ← 書き換える（入れないもの）

install, skipped = [], []
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if not line:
        continue
    name = re.split(r"[<>=!~\[;\s]", line)[0]
    (skipped if name.lower() in {s.lower() for s in SKIP} else install).append(line)
print("入れる:", install)
print("外す  :", skipped)

PIP_ARGS = " ".join(f'"{x}"' for x in install)  # >= などを含んでも壊れないよう、1つずつ引用符で囲む
if install:
    %pip install -q {PIP_ARGS}

## 5. 学習済みの重みを取得する（書き換える）

目安：約3分（うち計算 約1分）

README の指示どおり、重みを `checkpoints` フォルダに置きます。重みの多くは Hugging Face Hub（HF Hub）にあり、`hf_hub_download` で取得できます。README のダウンロードリンクが `https://huggingface.co/<リポジトリ>/resolve/main/<ファイル名>` の形なら、そこからリポジトリとファイル名がわかります。

**ゲート付きの重み**（SAM3 など、ページに「利用申請」のボタンがあるもの）は、次の準備が必要です。

1. HF Hub にログインし、モデルのページで利用を申請します（承認まで時間がかかることがあります）。
2. HF の Settings → Access Tokens で、Read のトークンを作ります。
3. Colab の左の「シークレット」（鍵のアイコン）に、名前 `HF_TOKEN` でトークンを登録し、「ノートブックからのアクセス」をオンにします。

登録すると、`hf_hub_download` がトークンを使って取得します。トークンはパスワードと同じなので、セルに直接書いたり、ノートブックを共有するときに含めたりしないでください。

In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS = [  # ← 書き換える（HF Hub のリポジトリ, ファイル名）
    ("depth-anything/Depth-Anything-V2-Small", "depth_anything_v2_vits.pth"),
]
WEIGHTS_DIR = "checkpoints"  # ← 書き換える（README が指定する置き場所）

for repo_id, filename in WEIGHTS:
    path = Path(hf_hub_download(repo_id, filename, local_dir=WEIGHTS_DIR))
    print(path, f"{path.stat().st_size / 2**20:.0f} MiB")

## 6. 公式の例を動かす（書き換える）

目安：約5分（うち計算 約1分）

まず README のコマンドを、そのまま動かします。自分で何か変えるのは、公式の例が動いてからにします（動かないとき、原因が自分の変更か、環境かを切り分けられるように）。

スクリプトの引数は、`--help` で一覧できます。

In [ ]:
!python run.py --help

In [ ]:
# README の "Running script on images" の例。--encoder だけ Small（vits）に変えた
!python run.py --encoder vits --img-path assets/examples --outdir outputs/examples

結果は `outputs/examples` に画像で保存されます。左が入力の写真、右が推定した奥行きで、赤いほど近く、青いほど遠いことを表します。

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def show_outputs(folder, n=4):
    """folder の画像を、先頭から n 枚表示する"""
    paths = sorted(Path(folder).glob("*.png"))[:n]
    fig, axes = plt.subplots(len(paths), 1, figsize=(10, 3.5 * len(paths)), squeeze=False)
    for ax, path in zip(axes[:, 0], paths):
        ax.imshow(Image.open(path))
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_outputs("outputs/examples")

## 7. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD = True` にして実行すると、アップロードのボタンが出ます。スマートフォンで撮った写真なども試してみましょう（JPEG か PNG にしてください。iPhone の HEIC は読めません）。`False` のままなら、例の画像を1枚使います。この画像（`demo05.jpg`）は手描きの線画ですが、線だけの絵からも、椅子が手前で壁が奥という奥行きが推定されます。

奥行きが遠くまで続く風景、近くに物がある室内、鏡やガラス、絵や写真を撮った写真など、うまくいきそうなものと、難しそうなものの両方を試すと、モデルの得意と不得意が見えてきます。

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる

MY_DIR = Path("my_images")
MY_DIR.mkdir(exist_ok=True)
if USE_UPLOAD:
    from google.colab import files
    for name, data in files.upload().items():
        (MY_DIR / name).write_bytes(data)
elif not any(MY_DIR.iterdir()):
    shutil.copy("assets/examples/demo05.jpg", MY_DIR)
print(sorted(p.name for p in MY_DIR.iterdir()))

In [ ]:
!python run.py --encoder vits --img-path my_images --outdir outputs/my_images
show_outputs("outputs/my_images")

## 8. Python から呼び出して応用する

目安：約10分（うち計算 約1分）

`run.py` は、README の "Use our models" に書かれた処理（モデルを作る → 重みを読む → `infer_image` で推定）を、フォルダの画像に順にかけて保存しているだけです。同じ処理を Python から呼ぶと、推定した奥行きを数値の配列として受け取り、別の処理につなげられます。

### 8-1. モデルを作って重みを読む

README のコードを写し、Small の設定だけ残しました。

In [ ]:
import cv2
import numpy as np

from depth_anything_v2.dpt import DepthAnythingV2

model_configs = {
    "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]},
}
ENCODER = "vits"

model = DepthAnythingV2(**model_configs[ENCODER])
model.load_state_dict(torch.load(f"checkpoints/depth_anything_v2_{ENCODER}.pth", map_location="cpu"))
model = model.to(device).eval()
print(f"パラメータの数: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")  # README の表と比べる

README の表（Depth-Anything-V2-Small：24.8M）と同じ数になれば、モデルの形と重みが合っています。

このモデルは、画像を 14×14 画素の小さな正方形（パッチ）に分け、各パッチを1つのトークンとして Transformer に入れます。入力の大きさ 518 なら、縦横 518 ÷ 14 = 37 個ずつ、37 × 37 = 1369 個のトークンの間でアテンションを計算しています。

### 8-2. 奥行きを推定する

In [ ]:
image_path = sorted(MY_DIR.iterdir())[0]
raw = cv2.imread(str(image_path))  # OpenCV は色を B, G, R の順で読む
depth = model.infer_image(raw)     # 縦×横の配列。値が大きいほど近い（相対的な値で、単位はない）
print("画像:", raw.shape, "/ 奥行き:", depth.shape, f"/ 値の範囲 {depth.min():.2f}〜{depth.max():.2f}")

### 8-3. 応用：奥行きを使って背景をぼかす

スマートフォンのポートレートモードのように、近いところだけ残して、遠いところをぼかします。`NEAR_RATIO` を変えて、くっきり残す範囲がどう変わるか見てみましょう。

In [ ]:
NEAR_RATIO = 0.3  # 近いほうから何割の画素を、くっきり残すか

threshold = np.quantile(depth, 1 - NEAR_RATIO)
mask = (depth >= threshold).astype(np.float32)
mask = cv2.GaussianBlur(mask, (0, 0), 5)[..., None]  # 境目をなめらかにする
blurred = cv2.GaussianBlur(raw, (0, 0), 15)
portrait = (raw * mask + blurred * (1 - mask)).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(raw[..., ::-1])  # B, G, R → R, G, B に並べ替えて表示する
axes[0].set_title("input")
axes[1].imshow(depth, cmap="Spectral_r")
axes[1].set_title("depth (red = near)")
axes[2].imshow(portrait[..., ::-1])
axes[2].set_title(f"background blur (near {NEAR_RATIO:.0%})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. コードを直して、変更を残す

目安：約10分

### 9-1. コードを直す

`run.py` の色の付け方（カラーマップ）を変えてみます。

1. 左の「ファイル」で `Depth-Anything-V2/run.py` をダブルクリックすると、右にエディタが開きます。
2. `'Spectral_r'` を探し（Ctrl+F）、`'inferno'` に書き換えます。
3. Ctrl+S で保存します。

手で直す代わりに、次のセルを実行しても同じ変更になります。

In [ ]:
path = Path("run.py")
path.write_text(path.read_text(encoding="utf-8").replace("'Spectral_r'", "'inferno'"), encoding="utf-8")
print("run.py を書き換えました")

`git diff` で、元のリポジトリからどこを変えたかを確かめます。`-` の行が元、`+` の行が変更後です。

In [ ]:
!git diff

In [ ]:
!python run.py --encoder vits --img-path my_images --outdir outputs/inferno
show_outputs("outputs/inferno")

### 9-2. 変更をパッチとして Google ドライブに残す

接続が切れると、直した `run.py` も消えます。`git diff` の出力をファイル（パッチ）にして Google ドライブに保存しておけば、次に開いたとき、3. で `RESTORE_PATCH = True` にするだけで同じ変更を当て直せます。

- パッチは小さなテキストファイルで、元のリポジトリから何を変えたかが一目でわかります。
- パッチは 2. の `REPO_REF` の版に対する差分です。`REPO_REF` を変えると、当て直せないことがあります。
- 初めて実行すると、Google ドライブへのアクセスの許可を求められます。許可の画面には、Google フォトや連絡先など、ドライブ以外の項目も並びます。これは Colab がドライブをつなぐ仕組みの都合で、一部の項目だけを選ぶとつながらずにエラー（`credential propagation was unsuccessful`）になります。**すべて選択**して続けてください。このノートブックが書き込むのは、ドライブの `seminar2_lesson02` フォルダのパッチだけです。

ほかの残し方（リポジトリごとドライブに置く、VS Code から Colab を使う）は、ガイドで比べています。

In [ ]:
if IN_COLAB and not Path("/content/drive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# git diff の出力（標準出力だけ。警告の文は混ぜない）を、そのままファイルにする
diff = subprocess.run(["git", "diff"], capture_output=True, check=True).stdout
if diff:
    PATCH_FILE.write_bytes(diff)
    print("保存しました:", PATCH_FILE, f"（{len(diff.splitlines())} 行）")
    !git diff --stat
else:
    print("変更がないので、保存しませんでした（前のパッチが残っていれば、それは消していません）")

## 10. 記録する

目安：約10分

自分で選んだリポジトリを動かしたら、下のセルをダブルクリックして書き込みます。うまく動かなかった場合も、どこまで進んで、何が起きたかを書いておくと、次に別のリポジトリを試すときの手がかりになります。

- **リポジトリ**（URL とコミット）：
- **何ができるモデルか**（1〜2文）：
- **ライセンス**（コードと重み）：
- **動かすまでに直したこと、つまずいたこと**（エラーの最後の行と、どう解決したか）：
- **試した入力と結果**（うまくいったもの、うまくいかなかったもの）：
- **研究や身の回りで、どう使えそうか**：